# Training an Indic Multilingual ASR Model

Companion notebook to the **Model Training** deck. Assumes the dataset produced
by `01_Data_Preparation.ipynb`: tarred shards per language, a unified tokenizer,
and an `input_cfg.yaml` combining them.

| Deck slide | Slide title | Notebook section |
|---|---|---|
| — | — | 0. Environment and base checkpoint |
| 2 | Encoder Transfer with Decoder Reinitialisation | 1. Encoder transfer |
| 3 | Choosing the Training Objective | 1b. Choosing the training objective |
| 4 | Prompt Modes for Code-Switched Input | 2. Prompt modes |
| 5 | Balancing Languages with Temperature Sampling | 3. Language balance |
| 6 | Two-Stage Curriculum: Coverage, Then Domain | 4. Two-stage curriculum |
| 7 | Selecting the Streaming Latency Operating Point | 5. Streaming latency |
| 8 | Biasing Domain Vocabulary Without Retraining | 6. Domain keyword boosting |
| 9 | Validating Code-Switched Performance | 7. Validation |
| 10 | Recommended Next Steps | 8. Checklist |

Runs on an NVIDIA Brev instance with the NeMo container. Sections requiring a
GPU or real data skip with an explicit message outside that environment.

---
## 0. Environment and base checkpoint

### Running this on NVIDIA Brev

Same Launchable configuration as the data preparation notebook: a GPU instance,
the NeMo container as a single container with the tag pinned from the current
NGC catalog, `NGC_API_KEY` as a launch parameter, and a Secure Link on port
`8888`.

Set Disk Storage with the base checkpoint and the tarred dataset in mind — both
are large, and the checkpoint is unpacked as well as downloaded.

The notebook verifies the environment it is running in rather than naming a
container image, so the same file works on Brev or on any equivalent instance.

### Preflight

In [ ]:
import importlib, shutil, subprocess, os, sys
from pathlib import Path

def _mod(n):
    try: importlib.import_module(n); return True
    except Exception: return False

def _gpu():
    if not shutil.which('nvidia-smi'): return None
    try:
        return [l.strip() for l in subprocess.run(
            ['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
            capture_output=True, text=True, check=True).stdout.strip().split('\n') if l.strip()]
    except Exception: return None

ENV = {
    'nemo'      : _mod('nemo.collections.asr'),
    'torch'     : _mod('torch'),
    'lhotse'    : _mod('lhotse'),
    'omegaconf' : _mod('omegaconf'),
    'sentencepiece': _mod('sentencepiece'),
    'kaldialign': _mod('kaldialign'),
    'huggingface_hub': _mod('huggingface_hub'),
}
GPUS = _gpu(); ENV['gpu'] = GPUS is not None

print(f"{'component':<18}status")
print('-'*40)
for k,v in ENV.items(): print(f"{k:<18}{'ok' if v else 'MISSING'}")
print()
for g in (GPUS or ['no GPU visible']): print('gpu:', g)

if not ENV['kaldialign']:
    subprocess.run(f'{sys.executable} -m pip install -q kaldialign', shell=True, check=False)
    ENV['kaldialign'] = _mod('kaldialign')
    print('kaldialign installed:', ENV['kaldialign'])

In [ ]:
SPEECH = Path(os.environ.get('SPEECH_REPO', '/workspace/Speech'))
DATA   = Path(os.environ.get('DATA_ROOT',  '/workspace/data'))
MODELS = Path(os.environ.get('MODEL_ROOT', '/workspace/models'))
CONF   = Path(os.environ.get('CONF_DIR',   '/workspace/conf'))
for p in (DATA, MODELS, CONF): p.mkdir(parents=True, exist_ok=True)

def ensure_speech_repo():
    if SPEECH.exists(): print('present:', SPEECH); return True
    if not shutil.which('git'): print('git unavailable'); return False
    r = subprocess.run(['git','clone','--depth','1',
                        'https://github.com/NVIDIA-NeMo/Speech.git', str(SPEECH)],
                       capture_output=True, text=True)
    print('cloned' if r.returncode==0 else f'failed: {r.stderr[-200:]}')
    return SPEECH.exists()

ENV['speech_repo'] = ensure_speech_repo()

### Base checkpoint

The checkpoint supplies the multilingual encoder. Fetching it needs network
access and, for gated repositories, credentials — supply those through the
Launchable's launch parameters rather than writing them into the notebook.

In [ ]:
BASE_REPO = os.environ.get('BASE_MODEL_REPO',
                           'nvidia/nemotron-3.5-asr-streaming-0.6b')
BASE_DIR  = MODELS / BASE_REPO.split('/')[-1]

def fetch_base_checkpoint():
    if any(BASE_DIR.glob('*.nemo')):
        print('already present:', BASE_DIR); return True
    if not ENV['huggingface_hub']:
        print('huggingface_hub unavailable; download the checkpoint manually')
        return False
    from huggingface_hub import snapshot_download
    try:
        snapshot_download(repo_id=BASE_REPO, local_dir=str(BASE_DIR),
                          allow_patterns=['*.nemo', '*.yaml', '*.json'])
        print('downloaded to', BASE_DIR); return True
    except Exception as e:
        print(f'download failed ({type(e).__name__}) - check access and credentials')
        return False

ENV['base_ckpt'] = fetch_base_checkpoint()
NEMO_FILE = next(iter(BASE_DIR.glob('*.nemo')), None)
print('checkpoint:', NEMO_FILE)

In [ ]:
# Read the configuration the checkpoint was trained with, before writing your own.
def show_base_config(keys=('encoder.att_context_size',
                           'encoder.subsampling_factor',
                           'encoder.d_model',
                           'optim.sched.name',
                           'tokenizer.type')):
    cfg_path = next(iter(BASE_DIR.glob('*.yaml')), None)
    if not (ENV['omegaconf'] and cfg_path):
        print('skipped: no config file available yet'); return None
    from omegaconf import OmegaConf
    cfg = OmegaConf.load(cfg_path)
    for k in keys:
        print(f'{k:<32} {OmegaConf.select(cfg, "model." + k)}')
    return cfg

BASE_CFG = show_base_config()

---
## 1. Encoder transfer

*Deck slide 2 — Encoder Transfer with Decoder Reinitialisation.*

A new tokenizer changes the output vocabulary, which makes the existing
decoder and joint invalid. The encoder is loaded from the base checkpoint;
the decoder and joint are reinitialised.

In [ ]:
TRANSFER = '''
init_from_nemo_model:
  model0:
    path: "/data/models/nemotron-3.5-asr-streaming-0.6b/nemotron-3.5-asr-streaming-0.6b.nemo"
    include: ['encoder']
    exclude: ['decoder', 'joint']
'''
print(TRANSFER)

Three consequences worth stating before the first run:

- **Initial validation error is uninformative.** A reinitialised decoder
  cannot transcribe. Checking that the first validation looks sane applies to
  fine-tuning, not to this.
- **Forgetting is not the risk.** No pretrained decoder remains to degrade.
  The risk is insufficient training of the new one.
- **The encoder is not frozen.** `include` and `exclude` control which
  weights are *loaded*, not which are trainable. Gradients flow into the
  encoder, which is part of why the long warmup matters — it protects a good
  encoder while the decoder becomes sane.

---
## 1b. Choosing the training objective

*Deck slide 3 — Choosing the Training Objective.*

The loss decides what the encoder learns and which decoder can be deployed.
Three routes, in increasing order of cost.

| Route | Trained with | Decodes with | Model class |
|---|---|---|---|
| Transducer only | transducer loss throughout | transducer | `EncDecRNNTBPEModel` |
| Transducer, then CTC | transducer first, CTC in a second stage | CTC | two runs, encoder transferred |
| Dual head | both objectives jointly, weighted | either, chosen at load | `EncDecHybridRNNTCTCBPEModel` |

The reference configuration is **transducer only**. The other two are
deliberate departures from it.

### Route 1 — transducer only

The prediction network conditions on output history, so alignment is learned
jointly with the label sequence. Highest accuracy, and the streaming
behaviour described in section 5 is a transducer property.

This is what the reference Indic configuration does, and what section 1
already set up.

### Route 2 — transducer first, then CTC

Two stages. The first trains under the transducer objective; the second takes
that encoder and trains a CTC head on top of it.

**Why the order matters.** Because the transducer conditions on output
history, the encoder is pushed toward context-dependent representations —
it cannot rely on the decoder being memoryless. A CTC head trained afterwards
inherits that encoder, then decodes in a single pass with no autoregressive
step. The intent is transducer-quality representations at CTC throughput.

CTC alone assumes conditional independence across frames, so training CTC from
the start does not produce the same encoder.

This is not a packaged configuration — it is two runs wired together by
encoder transfer, using the same mechanism as section 1.

In [ ]:
STAGE_A = '''
# Stage A - transducer objective
# examples/asr/speech_to_text_finetune.py with an RNNT config
init_from_nemo_model:
  model0:
    path: "<base checkpoint>.nemo"
    include: ['encoder']
    exclude: ['decoder', 'joint']
'''

STAGE_B = '''
# Stage B - CTC objective on the encoder trained in stage A
# examples/asr/speech_to_text_ctc_bpe.py with a CTC config
init_from_nemo_model:
  model0:
    path: "<stage A checkpoint>.nemo"
    include: ['encoder']          # the CTC head is new
'''
print(STAGE_A); print(STAGE_B)
print('Keep the tokenizer identical across both stages, or the CTC head')
print('is predicting a different vocabulary than the encoder was trained for.')

### Route 3 — dual head

One encoder, two output heads, both losses computed at every step and combined
by a weight. At inference one head is selected; the other contributes nothing
to latency.

The cost is training-side: two losses per step means slower steps and higher
memory.

In [ ]:
HYBRID = '''
model:
  # requires the hybrid model class rather than the RNNT one
  aux_ctc:
    ctc_loss_weight: 0.3          # balance between the two objectives
    use_cer: false
    ctc_reduction: mean_batch
    decoder:
      _target_: nemo.collections.asr.modules.ConvASRDecoder
      feat_in: null
      num_classes: -1
      vocabulary: []
'''
print(HYBRID)

In [ ]:
# Selecting the head at inference on a hybrid checkpoint
#
#   asr_model.change_decoding_strategy(decoder_type='ctc')    # throughput
#   asr_model.change_decoding_strategy(decoder_type='rnnt')   # accuracy
#
# Evaluate both on the same frozen manifest so the trade is measured
# rather than assumed.

def compare_heads(results):
    """results: {'ctc': (wer, rtfx), 'rnnt': (wer, rtfx)}"""
    print(f"{'head':<8}{'WER':>9}{'RTFx':>9}")
    print('-' * 26)
    for head, (wer, rtfx) in results.items():
        print(f'{head:<8}{wer:>8.2%}{rtfx:>9.0f}')
    if len(results) == 2:
        (h1, (w1, r1)), (h2, (w2, r2)) = results.items()
        print()
        print(f'{h2} vs {h1}: WER {w2-w1:+.2%}, throughput {r2/r1:.2f}x')

# Illustrative shape of the report; replace with measured values.
compare_heads({'rnnt': (0.121, 1450), 'ctc': (0.138, 2080)})

> **Choosing between the routes.** Transducer only unless there is a specific
> reason otherwise. Take route 2 when CTC throughput is a hard requirement and
> a separate model per operating point is acceptable. Take route 3 when one
> checkpoint has to serve both, and the training cost is acceptable.
>
> Note that the hybrid route interacts with section 5: the multi-context
> streaming behaviour is a transducer property, so verify it on whichever head
> is actually deployed.

---
## 2. Prompt modes

*Deck slide 4 — Prompt Modes for Code-Switched Input.*

The language is not declared at request time, so the model is trained to
operate without that information.

| Mode | Behaviour | Use for |
|---|---|---|
| `langID` | the true language identifier is always supplied | language-forced tasks |
| `auto` | a learned language-agnostic representation | code-switching |
| `unified` | alternates between the two during training | multilingual, default |

In [ ]:
# The prompt table needs one entry per target language plus the agnostic
# mode. A mismatch between num_prompts and the dictionary fails silently.

def prompt_block(langs):
    entries = {lang: i for i, lang in enumerate(langs)}
    entries['auto'] = len(langs)
    return {
        'prompt_dictionary': entries,
        'num_prompts': len(entries),
    }

LANGS = os.environ.get('TARGET_LANGS', 'hi,mr').split(',')
block = prompt_block(LANGS)
print(block)
assert block['num_prompts'] == len(block['prompt_dictionary'])

In [ ]:
PROMPT_CONFIG = '''
model:
  model_defaults:
    initialize_prompt_feature: true
    num_prompts: <one per language, plus auto>
    prompt_dictionary: <mapping produced above>

  train_ds:
    lang_field: lang
    prompt_mode_field: prompt_mode      # set per source via input_cfg tags
    default_prompt_mode: unified
    unified_auto_ratio: 0.5             # share of batches seeing agnostic
'''
print(PROMPT_CONFIG)

> **Open item.** Confirm what `validation_ds` inherits for prompt mode in the
> deployed version. If validation runs language-forced while deployment runs
> agnostic, the validation curve does not represent production behaviour.

---
## 3. Language balance

*Deck slide 5 — Balancing Languages with Temperature Sampling.*

At natural proportions the smallest source contributes almost nothing to the
gradient. Temperature sampling flattens the distribution: sources are drawn
in proportion to their natural share raised to the power `1/T`, renormalised.

In [ ]:
def sampling_distribution(hours_by_source, T):
    total = sum(hours_by_source.values())
    raw = {k: (v / total) ** (1.0 / T) for k, v in hours_by_source.items()}
    z = sum(raw.values())
    return {k: v / z for k, v in raw.items()}

def temperature_table(hours_by_source, temperatures=(1.0, 2.0, 3.0, 5.0)):
    names = list(hours_by_source)
    header = 'T'.ljust(6) + ''.join(n.rjust(12) for n in names)
    print(header); print('-' * len(header))
    for T in temperatures:
        d = sampling_distribution(hours_by_source, T)
        print(str(T).ljust(6) + ''.join(f'{d[n]:>11.1%} ' for n in names))

# Replace with the inventory produced in the data preparation notebook
temperature_table({'large': 1000.0, 'medium': 100.0, 'small': 10.0})

**Reading the table.** Low temperature leaves the smaller sources
effectively ignored. High temperature repeats limited audio until it is
memorised. Derive the value from actual per-language volumes rather than
carrying across a reference setting chosen for a different mixture.

In [ ]:
BALANCE_CONFIG = '''
model:
  train_ds:
    is_concat: true
    concat_sampling_technique: "temperature"
    concat_sampling_temperature: <derived above>
    bucketing_strategy: "synced_randomized"
'''
print(BALANCE_CONFIG)

---
## 4. Two-stage curriculum

*Deck slide 6 — Two-Stage Curriculum: Coverage, Then Domain.*

Stage one establishes language coverage on public corpora. Stage two
specialises to the domain, continuing from the stage-one checkpoint.
Retaining public corpora in stage two — reduced rather than removed — is the
safeguard against forgetting.

In [ ]:
def stage_weights(stage, public_sources, private_sources):
    """Produce the per-source weights for an input_cfg at each stage."""
    if stage == 1:
        pub, priv = 1.0, 0.2
    elif stage == 2:
        pub, priv = 0.2, 1.0
    else:
        raise ValueError('stage must be 1 or 2')
    w = {s: pub for s in public_sources}
    w.update({s: priv for s in private_sources})
    return w

PUBLIC  = ['indicvoices', 'kathbath', 'shrutilipi']
PRIVATE = ['client_domain']

for stage in (1, 2):
    print(f'stage {stage}: {stage_weights(stage, PUBLIC, PRIVATE)}')

In [ ]:
STAGE_1 = '''
# Establish language coverage: encoder transferred, decoder reinitialised
init_from_nemo_model:
  model0:
    path: "<base checkpoint>.nemo"
    include: ['encoder']
    exclude: ['decoder', 'joint']
'''

STAGE_2 = '''
# Specialise to the domain: continue from stage 1, tokenizer unchanged,
# so the decoder is retained rather than reinitialised
init_from_nemo_model:
  model0:
    path: "<stage 1 checkpoint>.nemo"
'''
print(STAGE_1); print(STAGE_2)

In [ ]:
# Launch (either stage)
#   python $SPEECH/examples/asr/speech_to_text_finetune.py \
#     --config-path=$CONF_DIR --config-name=<stage config> \
#     model.train_ds.num_workers=2 \
#     model.train_ds.max_duration=30.0

print('Lhotse produces an unbounded stream, so there is no natural epoch.',
      'Training length is governed by max_steps, and limit_train_batches',
      'defines the interval used for validation.')

---
## 5. Streaming latency

*Deck slide 7 — Selecting the Streaming Latency Operating Point.*

Several attention context sizes are trained jointly, so one checkpoint serves
several latency operating points chosen at inference. Training a single
context fixes one latency for the lifetime of the checkpoint.

In [ ]:
def lookahead_ms(right_context, subsampling_factor=8, window_stride_s=0.01):
    """One encoder frame spans subsampling_factor * window_stride seconds."""
    frame_ms = subsampling_factor * window_stride_s * 1000
    return right_context * frame_ms

CONTEXTS = [[56, 13], [56, 6], [56, 3], [56, 1], [56, 0]]
print(f"{'left':>6}{'right':>7}{'lookahead':>12}")
print('-' * 25)
for left, right in CONTEXTS:
    print(f'{left:>6}{right:>7}{lookahead_ms(right):>10.0f} ms')

In [ ]:
LATENCY_CONFIG = '''
model:
  encoder:
    subsampling_factor: 8
    causal_downsampling: true
    conv_context_size: causal
    att_context_size: [[56,13],[56,6],[56,3],[56,1],[56,0]]
'''
print(LATENCY_CONFIG)
print()
print('causal_downsampling and a causal convolution context are what make',
      'the zero-lookahead setting genuinely causal.')

---
## 6. Domain keyword boosting

*Deck slide 8 — Biasing Domain Vocabulary Without Retraining.*

Applied to an already trained checkpoint at decoding time. The supplied
phrase list is compiled into a prefix tree — each node a token, each complete
path one phrase. During decoding a hypothesis that begins matching a path
receives a score bonus that grows with depth, and loses it if the path breaks.

One approach covers CTC, RNN-T and TDT. Material predating it describes a
different path per decoder and is out of date.

In [ ]:
# The phrase list is the domain vocabulary identified during data preparation:
# product names, account terminology, branch names, regulatory vocabulary.
# One phrase per line.

PHRASES = Path('/data/boosting/key_phrases.txt')

def write_phrase_list(terms, path=PHRASES, capitalised_model=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    out = sorted({t.strip() for t in terms if t.strip()})
    if capitalised_model:
        # Models with native capitalisation require phrases capitalised in
        # advance, including full-word capitalisation for abbreviations.
        out = sorted({t.upper() if t.isupper() else t.title() for t in out})
    path.write_text('\n'.join(out) + '\n')
    print(f'{len(out)} phrases -> {path}')
    return out

# write_phrase_list(['<domain term>', '<product name>', '<branch name>'])

In [ ]:
# Optionally pre-build the tree; otherwise it is built from the phrase list
# at decoding time.
#
#   python $SPEECH/scripts/asr_context_biasing/build_gpu_boosting_tree.py \
#     --key_phrases_file /data/boosting/key_phrases.txt \
#     --output_path /data/boosting/tree

# Decode with boosting through the standard evaluation entry point.
#
#   python $SPEECH/examples/asr/speech_to_text_eval.py \
#     model_path=<checkpoint>.nemo \
#     dataset_manifest=<test manifest>.json \
#     rnnt_decoding.strategy=greedy_batch \
#     boosting_tree.key_phrases_file=/data/boosting/key_phrases.txt \
#     boosting_tree.context_score=1.0 \
#     boosting_tree.depth_scaling=2.0

print('Tuning surface: the size of the bonus, and how strongly it grows',
      'with depth. Set too high, the model emits phrases never spoken.')

### Measure the effect on the phrase list, not on overall error rate

Boosting is intended to move accuracy on a specific vocabulary. Aggregate
word error rate will barely register it. NeMo ships the keyword scorer, so
there is no need to write one.

In [ ]:
# python $SPEECH/scripts/asr_context_biasing/compute_key_words_fscore.py \
#   --input_manifest <decoded manifest>.json \
#   --key_words_file /data/boosting/key_phrases.txt

print('Report the baseline and boosted keyword scores side by side,',
      'together with the change in real-time factor.')

> **Caution.** The production serving stack uses a separate implementation
> with different parameter ranges and different behaviour on out-of-vocabulary
> terms. Values tuned here do not transfer; assuming they do produces silent
> over-boosting.

---
## 7. Validation

*Deck slide 9 — Validating Code-Switched Performance.*

Evaluate twice: once with the language declared, once without. The difference
between the two is the measurement.

In [ ]:
# Language-forced
#   python $SPEECH/examples/asr/transcribe_speech_parallel.py \
#     model=<checkpoint>.nemo \
#     predict_ds.manifest_filepath=<test manifest>.json \
#     predict_ds.batch_size=32 \
#     output_path=/data/results/<run>/declared
#
# Language-agnostic: same command with the prompt mode set to the agnostic
# entry, writing to a separate output path.

print('Both runs must use the same frozen test manifest.')

In [ ]:
if ENV['nemo']:
    from nemo.collections.asr.metrics.wer import word_error_rate

    def score(references, hypotheses):
        return {
            'wer': word_error_rate(hypotheses=hypotheses, references=references),
            'cer': word_error_rate(hypotheses=hypotheses, references=references,
                                   use_cer=True),
            'n':   len(references),
        }

    demo_ref = ['the account balance is twelve thousand five hundred']
    demo_hyp = ['the account balance is twelve thousand five hundred rupees']
    print('smoke test:', score(demo_ref, demo_hyp))
else:
    print('skipped: scoring uses the NeMo WER metric')

Report per language rather than aggregated — a mean across languages hides
the ones that are failing. Include utterance counts alongside every rate.

The gap also has a commercial reading: it quantifies what a language-detection
stage ahead of the model would be worth.

In [ ]:
# Inspect individual errors rather than only the summary
#   python $SPEECH/tools/speech_data_explorer/data_explorer.py \
#     /data/results/<run>/predictions_all.json --port 8001

---
## 8. Checklist

*Deck slide 10 — Recommended Next Steps.*

Confirm each item before a long run.

In [ ]:
CHECKLIST = [
    'Prompt table sized for every target language plus the agnostic mode',
    'Prompt mode set per source in input_cfg, not only globally',
    'Sampling temperature derived from actual per-language volumes',
    'Stage one and stage two source weights defined',
    'Stage two initialises from the stage one checkpoint, decoder retained',
    'Attention context list covers every latency the deployment may need',
    'Bucketing strategy synced across distributed ranks',
    'Validation prompt mode confirmed against the deployment condition',
    'Phrase list assembled from the domain vocabulary',
    'Keyword score reported alongside error rates',
    'Boosting parameters not carried across from the serving stack',
]
for i, item in enumerate(CHECKLIST, 1):
    print(f'{i:>3}. [ ] {item}')

### References

| Repository | Contents |
|---|---|
| `bgiddwani-ai/multilingual_nemo_asr` | Indic reference pipeline and training configuration |
| `NVIDIA-NeMo/Speech` | training, context biasing, evaluation, data explorer |
| `huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b` | base checkpoint |

**Suggested first milestone.** A single language pair trained end to end,
with the declared-versus-agnostic difference measured. Not the full language
set at once.